# Claude Code CLI

Test the coding assistant using **Claude Code** (terminal-based agent) connected to MaaS.

This notebook will:
1. Discover your MaaS endpoints dynamically
2. Verify connectivity
3. Generate environment variables and configuration for Claude Code
4. Guide you through CLI testing with expected screenshots

**Prerequisites:** Phases 0–3 completed

## Step 1: Discover Endpoints

In [1]:
import os
import subprocess
import json
import urllib.request
import ssl
from dotenv import load_dotenv

load_dotenv()

# Try .env first, then fall back to oc CLI
CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN")
if not CLUSTER_DOMAIN:
    result = subprocess.run(
        ["oc", "get", "ingresses.config.openshift.io", "cluster",
         "-o", "jsonpath={.spec.domain}"],
        capture_output=True, text=True
    )
    if result.returncode == 0 and result.stdout.strip():
        CLUSTER_DOMAIN = result.stdout.strip()
    else:
        raise RuntimeError(
            "Cannot discover cluster domain. "
            "Set CLUSTER_DOMAIN in .env or run 'oc login' first."
        )

MAAS_HOST = f"https://maas-api.{CLUSTER_DOMAIN}"

token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
OC_TOKEN = token_result.stdout.strip()

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

# Get models
req = urllib.request.Request(
    f"{MAAS_HOST}/v1/models",
    headers={"Authorization": f"Bearer {OC_TOKEN}"}
)
try:
    with urllib.request.urlopen(req, context=ctx) as resp:
        models_data = json.loads(resp.read())
    MODEL_NAME = models_data["data"][0]["id"]
except Exception as e:
    MODEL_NAME = "MODEL_NOT_FOUND"

# Get MCP servers
routes_result = subprocess.run(
    ["oc", "get", "httproute", "-n", "mcp-servers",
     "-l", "maas.opendatahub.io/managed=true",
     "-o", "jsonpath={range .items[*]}{.metadata.name}\n{end}"],
    capture_output=True, text=True
)
mcp_servers = {}
for line in routes_result.stdout.strip().split("\n"):
    if line:
        name = line.replace("mcp-route-", "")
        mcp_servers[name] = f"{MAAS_HOST}/mcp/{name}/mcp"

print(f"MaaS Endpoint  : {MAAS_HOST}/v1")
print(f"Model          : {MODEL_NAME}")
print(f"MCP Servers    : {len(mcp_servers)}")
print(f"Source         : {'.env' if os.getenv('CLUSTER_DOMAIN') else 'oc CLI'}")
print()
for name, url in mcp_servers.items():
    print(f"  {name:25s} -> {url}")

MaaS Endpoint  : https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/v1
Model          : qwen25-coder-7b
MCP Servers    : 6
Source         : .env

  code-sandbox              -> https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/code-sandbox/mcp
  context7                  -> https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/context7/mcp
  gh-grep                   -> https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/gh-grep/mcp
  github                    -> https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/github/mcp
  playwright                -> https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/playwright/mcp
  sequential-thinking       -> https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/sequential-thinking/mcp


## Step 2: Verify Connectivity

In [2]:
print("Connectivity Check:")
print("=" * 60)

try:
    test_req = urllib.request.Request(
        f"{MAAS_HOST}/v1/models",
        headers={"Authorization": f"Bearer {OC_TOKEN}"}
    )
    with urllib.request.urlopen(test_req, context=ctx) as resp:
        print(f"  [OK] Model endpoint - HTTP {resp.status}")
except Exception as e:
    print(f"  [FAIL] Model endpoint - {e}")

# Quick inference test
try:
    chat_body = json.dumps({
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "Say hello in one word."}],
        "max_tokens": 10
    }).encode()
    chat_req = urllib.request.Request(
        f"{MAAS_HOST}/v1/chat/completions",
        data=chat_body,
        headers={"Authorization": f"Bearer {OC_TOKEN}", "Content-Type": "application/json"}
    )
    with urllib.request.urlopen(chat_req, context=ctx, timeout=30) as resp:
        chat_data = json.loads(resp.read())
    reply = chat_data["choices"][0]["message"]["content"]
    print(f"  [OK] Inference test - Model replied: \"{reply.strip()}\"")
except Exception as e:
    print(f"  [WARN] Inference test - {e}")

Connectivity Check:


  [OK] Model endpoint - HTTP 200


  [OK] Inference test - Model replied: "Hi."


## Step 3: Generate Claude Code Configuration

Claude Code uses environment variables for OpenAI-compatible endpoints.

In [3]:
API_KEY = os.getenv("MAAS_API_KEY", "sk-oai-YOUR-KEY")

print("=" * 60)
print("ENVIRONMENT VARIABLES (add to ~/.bashrc or ~/.zshrc)")
print("=" * 60)
print(f'export OPENAI_BASE_URL="{MAAS_HOST}/v1"')
print(f'export OPENAI_API_KEY="{API_KEY}"')
print()
print("=" * 60)
print("CLAUDE CODE MCP CONFIG (~/.claude/claude_desktop_config.json)")
print("=" * 60)

claude_mcp_config = {
    "mcpServers": {
        name: {
            "type": "streamableHttp",
            "url": url,
            "headers": {"Authorization": f"Bearer {API_KEY}"}
        }
        for name, url in mcp_servers.items()
    }
}
print(json.dumps(claude_mcp_config, indent=2))

ENVIRONMENT VARIABLES (add to ~/.bashrc or ~/.zshrc)
export OPENAI_BASE_URL="https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/v1"
export OPENAI_API_KEY="sk-oai-YOUR-KEY"

CLAUDE CODE MCP CONFIG (~/.claude/claude_desktop_config.json)
{
  "mcpServers": {
    "code-sandbox": {
      "type": "streamableHttp",
      "url": "https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/code-sandbox/mcp",
      "headers": {
        "Authorization": "Bearer sk-oai-YOUR-KEY"
      }
    },
    "context7": {
      "type": "streamableHttp",
      "url": "https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/context7/mcp",
      "headers": {
        "Authorization": "Bearer sk-oai-YOUR-KEY"
      }
    },
    "gh-grep": {
      "type": "streamableHttp",
      "url": "https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/gh-grep/mcp",
      "headers": {
        "Authorization": "Bearer sk-oai-YOUR-KEY"
      }
    },
    "github": {
      "type": "st

## Step 4: Install Claude Code

In [4]:
%%bash
echo "Checking Claude Code installation..."
if command -v claude &> /dev/null; then
    echo "  [OK] Claude Code is installed: $(claude --version 2>/dev/null || echo 'version unknown')"
else
    echo "  [NOT INSTALLED] Install with:"
    echo "    npm install -g @anthropic-ai/claude-code"
fi

Checking Claude Code installation...


  [NOT INSTALLED] Install with:
    npm install -g @anthropic-ai/claude-code


## Step 5: Test Claude Code Agent

### 5a. Basic Code Generation

Run in your terminal:
```bash
cd /path/to/your/project
claude "Create a Dockerfile for a Python 3.11 FastAPI app with multi-stage build"
```

**Expected:** Claude Code generates a production-ready Dockerfile.

![Claude Code Generation](screenshots/12-claude-code-generate.png)

### 5b. Agent Workflow (file analysis)

```bash
claude "Review the current directory structure and suggest improvements"
```

**Expected:** Claude reads the file tree, analyzes structure, provides suggestions.

![Claude Code Agent](screenshots/13-claude-code-agent.png)

### 5c. MCP Tool Integration

```bash
claude "Use GitHub to list recent commits in this repository"
```

**Expected:** Claude uses the GitHub MCP server to fetch commit history.

![Claude Code MCP](screenshots/14-claude-code-mcp.png)

## Step 6: Verification Summary

In [5]:
print("Claude Code CLI - Verification Checklist")
print("=" * 60)
print(f"  Model Endpoint : {MAAS_HOST}/v1")
print(f"  Model Name     : {MODEL_NAME}")
print(f"  MCP Servers    : {len(mcp_servers)} configured")
print()
print("  [ ] Claude Code installed (npm install -g @anthropic-ai/claude-code)")
print("  [ ] Environment variables set (OPENAI_BASE_URL, OPENAI_API_KEY)")
print("  [ ] MCP config applied (~/.claude/claude_desktop_config.json)")
print("  [ ] Code generation works")
print("  [ ] Agent file analysis works")
print("  [ ] MCP tool calling works")
print()
print("Add screenshots to 6_ide_integration_test/screenshots/")

Claude Code CLI - Verification Checklist
  Model Endpoint : https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/v1
  Model Name     : qwen25-coder-7b
  MCP Servers    : 6 configured

  [ ] Claude Code installed (npm install -g @anthropic-ai/claude-code)
  [ ] Environment variables set (OPENAI_BASE_URL, OPENAI_API_KEY)
  [ ] MCP config applied (~/.claude/claude_desktop_config.json)
  [ ] Code generation works
  [ ] Agent file analysis works
  [ ] MCP tool calling works

Add screenshots to 6_ide_integration_test/screenshots/


## Troubleshooting

| Issue | Fix |
|-------|-----|
| Connection refused | Check `OPENAI_BASE_URL` - run Step 1 for correct URL |
| Auth error | Verify API key with `curl -H 'Authorization: Bearer $KEY' $URL/v1/models` |
| Timeout | Increase timeout: `claude --timeout 60000 "..."` |
| Model mismatch | Confirm model name matches InferenceService (`oc get isvc`) |
| SSL issues | Set `NODE_TLS_REJECT_UNAUTHORIZED=0` (dev only) |